# Tutorial 4: Principal Component Analysis (PCA) from Scratch

**Course**: Statistics for DSAI  
**Duration**: 1.5 hours  
**Dataset**: Iris Dataset

---

## What is PCA?

PCA (Principal Component Analysis) is a **dimensionality reduction** technique that:
- Finds new axes (principal components) that capture maximum variance
- Transforms data from high-dimensional space to lower dimensions
- Is **unsupervised** (doesn't use class labels)

Think of it as: **rotating your coordinate system** to align with the directions of maximum spread in your data.

---

## Learning Objectives

By the end of this tutorial, you will understand:

1. **Data as a matrix** - Representing data for linear algebra
2. **Mean centering** - Why zero-mean data is essential
3. **Covariance** - Measuring how features vary together
4. **Eigenvalues & eigenvectors** - The math behind PCA
5. **Explained variance** - How much info each component captures
6. **Projection** - Transforming data to PCA space
7. **Visualization** - Seeing high-dimensional data in 2D
8. **Limitations** - When PCA fails

---

## 1. Data as a Matrix

### Theory

In PCA, we treat our dataset as a **matrix**:

$$X \in \mathbb{R}^{n \times d}$$

Where:
- **n** = number of samples (rows) = 150 for Iris
- **d** = number of features (columns) = 4 for Iris

**Key insight**: Each row is one data point, each column is one feature.

### Why This Matters
- PCA uses **linear algebra** (matrix operations)
- All operations are done on the entire dataset at once
- This is called **vectorized** computation (efficient!)

---

In [ ]:
# =============================================================================
# SECTION 1: DATA AS A MATRIX
# =============================================================================

# Import necessary libraries
import numpy as np              # For numerical computations
import matplotlib.pyplot as plt # For visualization
from sklearn.datasets import load_iris  # Load sample dataset

# Load the Iris dataset
# Iris has 150 samples of 3 flower species
# Each sample has 4 features: sepal length, sepal width, petal length, petal width
iris = load_iris()

# Extract feature matrix (not including target labels)
X = iris.data  # Shape: (150, 4)

# Display dataset information
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"\nShape of X: {X.shape}")
print(f"  - {X.shape[0]} samples (rows)")
print(f"  - {X.shape[1]} features (columns)")
print(f"\nFeature names:")
for i, name in enumerate(iris.feature_names):
    print(f"  {i+1}. {name}")

print(f"\nTarget classes (flower species):")
for i, name in enumerate(iris.target_names):
    count = np.sum(iris.target == i)
    print(f"  {i+1}. {name}: {count} samples")

# Show first 5 data points
print("\nFirst 5 samples:")
print(X[:5])

---

## 2. Mean Centering

### Theory

**Mean centering** means subtracting the mean from each feature:

$$X_{centered} = X - \mu$$

Where $\mu$ is the vector of feature means.

### Why Do We Need This?

1. **Covariance interpretation**: Covariance measures spread *around the mean*. If data isn't centered, covariance is meaningless.

2. **Eigenvalue decomposition**: The math works best when data is centered at origin.

3. **Interpretation**: After centering, each feature has mean = 0.

### Analogy
Imagine measuring heights of people. If you don't subtract the average height, your variance calculation includes the average height itself, not the actual spread!

---

In [ ]:
# =============================================================================
# SECTION 2: MEAN CENTERING
# =============================================================================

# Step 2a: Compute the mean of each feature (column)
# axis=0 means we compute mean along rows (for each column)
# Result: array of 4 means (one for each feature)

means = np.mean(X, axis=0)

print("=" * 60)
print("STEP 2a: COMPUTE FEATURE MEANS")
print("=" * 60)
print("\nFormula: μ_j = (1/n) * Σ x_ij")
print("\nFeature means:")
for i, (name, mean_val) in enumerate(zip(iris.feature_names, means)):
    print(f"  {name}: {mean_val:.4f} cm")

# Step 2b: Center the data by subtracting the mean
# Broadcasting: 'means' is (1,4) but X is (150,4)
# NumPy automatically subtracts the same mean from each row

X_centered = X - means

print("\n" + "=" * 60)
print("STEP 2b: CENTER THE DATA")
print("=" * 60)
print("\nFormula: X_centered = X - μ")

# Verify centering worked - means should now be ~0
centered_means = np.mean(X_centered, axis=0)
print("\nVerification - means after centering (should be ≈ 0):")
print(f"  {centered_means}")

# Show the effect of centering on first feature
print("\nEffect on first feature (sepal length):")
print(f"  Before: min={X[:,0].min():.1f}, max={X[:,0].max():.1f}, mean={X[:,0].mean():.2f}")
print(f"  After:  min={X_centered[:,0].min():.1f}, max={X_centered[:,0].max():.1f}, mean={X_centered[:,0].mean():.2f}")

---

## 3. Covariance Matrix

### Theory

**Covariance** measures how two features change together:

$$Cov(x, y) = \frac{1}{n-1} \sum_{i=1}^{n} (x_i - \mu_x)(y_i - \mu_y)$$

### Interpretation
| Covariance Value | Meaning |
|------------------|--------|
| Positive | Features increase together |
| Negative | One increases, other decreases |
| Zero | No linear relationship |

### Covariance Matrix

For d features, we build a d×d matrix:

$$C = \frac{1}{n-1} X_{centered}^T X_{centered}$$

**Structure**:
- **Diagonal**: Variance of each feature
- **Off-diagonal**: Covariance between feature pairs

### Why PCA Needs This
PCA finds directions that maximize variance. The covariance matrix contains ALL the variance information!

---

In [ ]:
# =============================================================================
# SECTION 3: COVARIANCE MATRIX
# =============================================================================

# Get number of samples
n = X_centered.shape[0]

# Compute covariance matrix
# Formula: C = (1/(n-1)) * X^T * X
# This is equivalent to: (X_centered.T @ X_centered) / (n-1)

cov_matrix = (X_centered.T @ X_centered) / (n - 1)

print("=" * 60)
print("COVARIANCE MATRIX COMPUTATION")
print("=" * 60)
print("\nFormula: C = (1/(n-1)) * X^T * X")
print(f"\nShape: {cov_matrix.shape} (4 features × 4 features)")

# Display with labels
print("\nCovariance Matrix:")
feature_short = [name.replace(' (cm)', '') for name in iris.feature_names]
print("        ", "  ".join(f"{s:>8}" for s in feature_short))
for i, row in enumerate(cov_matrix):
    print(f"{feature_short[i]:>8}:", "  ".join(f"{v:>8.2f}" for v in row))

# Key observations
print("\n" + "-" * 60)
print("KEY OBSERVATIONS:")
print("-" * 60)

# Diagonal = variances
variances = np.diag(cov_matrix)
print("\n1. DIAGONAL = Variances (spread of each feature):")
for name, var in zip(feature_short, variances):
    print(f"   {name}: {var:.2f}")

# Off-diagonal = covariances
print("\n2. OFF-DIAGONAL = Covariances (relationships):")
print(f"   Petal length ↔ Petal width: {cov_matrix[2,3]:.2f} (high positive = grow together!)")
print(f"   Sepal length ↔ Sepal width: {cov_matrix[0,1]:.2f} (low)")

# Verify with numpy's built-in function
cov_verify = np.cov(X_centered.T)
print("\n3. Verification (np.cov):", np.allclose(cov_matrix, cov_verify))

---

## 4. Eigenvalues & Eigenvectors

### Theory

This is the **core of PCA**! The eigenvalue equation:

$$Cv = \lambda v$$

Where:
- **C** = covariance matrix (4×4)
- **v** = eigenvector (a direction)
- **λ** = eigenvalue (a number)

### What Does This Mean?

When we multiply the covariance matrix by its eigenvector:
- We get the same direction back
- Just stretched by the eigenvalue

### In PCA
- Each **eigenvector** = one principal component (a new axis direction)
- Each **eigenvalue** = variance along that direction

### Key Property
Eigenvectors are **orthogonal** (perpendicular) - they point in completely different directions!

---

In [ ]:
# =============================================================================
# SECTION 4: EIGENVALUE DECOMPOSITION
# =============================================================================

# Compute eigenvalues and eigenvectors
# np.linalg.eig returns complex numbers for non-symmetric matrices
# But covariance matrix is symmetric, so eigenvalues are real!

eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)

# Take real part (imaginary part should be ~0 anyway)
eigenvalues = np.real(eigenvalues)
eigenvectors = np.real(eigenvectors)

print("=" * 60)
print("EIGENVALUE DECOMPOSITION")
print("=" * 60)

print("\nFormula: C * v = λ * v")
print("\n--- EIGENVALUES ---")
print("(These represent variance in each direction)")

for i, ev in enumerate(eigenvalues):
    print(f"  λ{i+1} = {ev:.4f}")

print("\n--- EIGENVECTORS ---")
print("(Each column is one eigenvector - a direction in 4D space)")
print("\nMatrix form (each COLUMN is one eigenvector):")
print(np.round(eigenvectors, 4))

# Sort eigenvalues in descending order
# PCA wants biggest variance first!
sorted_idx = np.argsort(eigenvalues)[::-1]  # [::-1] reverses the array
eigenvalues = eigenvalues[sorted_idx]
eigenvectors = eigenvectors[:, sorted_idx]

print("\n" + "=" * 60)
print("SORTED EIGENVALUES (descending)")
print("=" * 60)
print("\nλ1 ≥ λ2 ≥ λ3 ≥ λ4 (this is the PCA convention)")
for i, ev in enumerate(eigenvalues):
    print(f"  λ{i+1} = {ev:.4f}")

print("\n--- EIGENVECTOR INTERPRETATION ---")
print("PC1 = eigenvector with largest eigenvalue")
print("PC2 = eigenvector with second largest eigenvalue")
print("etc.")

---

## 5. Explained Variance

### Theory

Now we know eigenvalues, but what do they mean?

**Explained Variance Ratio (EVR)**:

$$EVR_i = \frac{\lambda_i}{\sum_{j=1}^{d} \lambda_j}$$

This tells us what **percentage** of total variance each component captures.

### Cumulative Variance
$$Cumulative_k = \sum_{i=1}^{k} EVR_i$$

### The 95% Rule
We keep enough components to explain at least 95% of variance:

$$\sum_{i=1}^{k} EVR_i \geq 0.95$$

### Why 95%?
- 100% = original data (no reduction)
- 95% = good balance between compression and information
- Less = too much information loss

---

In [ ]:
# =============================================================================
# SECTION 5: EXPLAINED VARIANCE
# =============================================================================

# Total variance (sum of all eigenvalues)
total_variance = np.sum(eigenvalues)

# Explained variance ratio for each component
explained_variance_ratio = eigenvalues / total_variance

# Cumulative explained variance
cumulative_variance = np.cumsum(explained_variance_ratio)

print("=" * 60)
print("EXPLAINED VARIANCE ANALYSIS")
print("=" * 60)

print("\nFormula: EVR_i = λ_i / Σλ")

print("\n{:<8} {:>12} {:>12} {:>12}".format(
    "Component", "Eigenvalue", "Variance %", "Cumulative %"))
print("-" * 48)

for i in range(4):
    print("{:<8} {:>12.2f} {:>11.1f}% {:>11.1f}%".format(
        f"PC{i+1}", 
        eigenvalues[i], 
        explained_variance_ratio[i] * 100,
        cumulative_variance[i] * 100))

# Find number of components for 95%
n_components_95 = np.argmax(cumulative_variance >= 0.95) + 1

print("\n" + "=" * 60)
print("DECISION: HOW MANY COMPONENTS?")
print("=" * 60)
print(f"\n→ With 1 component: {cumulative_variance[0]*100:.1f}% variance")
print(f"→ With 2 components: {cumulative_variance[1]*100:.1f}% variance")
print(f"→ With 3 components: {cumulative_variance[2]*100:.1f}% variance")
print(f"\nFor 95% threshold: need {n_components_95} components")

In [ ]:
# Visualize explained variance
fig, ax = plt.subplots(figsize=(10, 5))

# Bar chart for individual variance
bars = ax.bar(range(1, 5), explained_variance_ratio * 100, 
              color='steelblue', alpha=0.7, label='Individual')

# Line for cumulative variance
ax.plot(range(1, 5), cumulative_variance * 100, 'ro-', 
        linewidth=2, markersize=8, label='Cumulative')

# 95% threshold line
ax.axhline(y=95, color='green', linestyle='--', linewidth=2, 
           label='95% threshold')

# Add value labels on bars
for i, (bar, val) in enumerate(zip(bars, explained_variance_ratio)):
    ax.text(bar.get_x() + bar.get_width()/2, val*100 + 2, 
            f'{val*100:.1f}%', ha='center', fontsize=10)

ax.set_xlabel('Principal Component', fontsize=12)
ax.set_ylabel('Explained Variance (%)', fontsize=12)
ax.set_title('Variance Explained by Each Principal Component', fontsize=14)
ax.legend(loc='center right')
ax.set_xticks(range(1, 5))
ax.set_ylim(0, 110)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ PC1 + PC2 capture 97.8% of variance - we can use just 2D!")

---

## 6. Projection onto Principal Components

### Theory

Now we transform our data to the new PCA coordinate system:

$$Z = X_{centered} \times W$$

Where:
- $X_{centered}$ = centered data (150 × 4)
- $W$ = eigenvector matrix (4 × k) where k = number of components
- $Z$ = transformed data (150 × k)

### What This Does
This is a **linear transformation** (matrix multiplication):
- We rotate the data to align with the new axes
- Each data point gets new coordinates in PCA space

### Analogy
Think of rotating a photograph:
- Original: axes are horizontal/vertical
- After PCA: axes point along the longest/shortest dimensions

---

In [ ]:
# =============================================================================
# SECTION 6: PROJECTION
# =============================================================================

# Get the eigenvector matrix (principal components)
# Each COLUMN is one eigenvector

W = eigenvectors  # Shape: (4, 4)

print("=" * 60)
print("PROJECTION: TRANSFORM TO PCA SPACE")
print("=" * 60)

print("\nFormula: Z = X_centered × W")
print("\nWhere W = eigenvector matrix (principal components)")
print(f"Shape: {W.shape} (4 features × 4 PCs)")

# Project ALL data (using all 4 components)
# This rotates the data to align with principal component axes
X_pca = X_centered @ W

print(f"\nOriginal data shape: {X.shape}")
print(f"PCA transformed shape: {X_pca.shape}")

# Verify: variance in PCA space should match eigenvalues
pca_variance = np.var(X_pca, axis=0, ddof=1)

print("\n" + "-" * 60)
print("VERIFICATION")
print("-" * 60)
print("\nVariance in PCA space vs Original eigenvalues:")
print("{:<12} {:>15} {:>15}".format("Component", "PCA Variance", "Eigenvalue"))
print("-" * 45)
for i in range(4):
    print("{:<12} {:>15.2f} {:>15.2f}".format(
        f"PC{i+1}", pca_variance[i], eigenvalues[i]))

print("\n✓ They match! (small difference due to floating point)")

# Now project with only 2 components (for visualization)
W_2d = eigenvectors[:, :2]  # Take first 2 eigenvectors
X_pca_2d = X_centered @ W_2d

print(f"\nWith 2D projection: {X_pca_2d.shape}")
print(f"Variance captured: {cumulative_variance[1]*100:.1f}%")

---

## 7. PCA Visualization

### Theory

One of PCA's most common uses is **visualization**:

- Reduce 4D data → 2D scatter plot
- Color points by their class (species)
- See if classes cluster together

### What to Look For
- Good separation = classes are distinct in PCA space
- Overlap = classes are difficult to separate

### Teaching Point
This visualization helps students understand that PCA might NOT perfectly separate classes (see next section on limitations!)

---

In [ ]:
# =============================================================================
# SECTION 7: VISUALIZATION
# =============================================================================

print("=" * 60)
print("PCA VISUALIZATION")
print("=" * 60)

# Create figure
fig, ax = plt.subplots(figsize=(10, 7))

# Define colors for each class
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']  # Blue, Orange, Green
markers = ['o', 's', '^']  # Circle, Square, Triangle

# Plot each class
for class_idx, (color, marker, name) in enumerate(zip(colors, markers, iris.target_names)):
    # Get indices for this class
    mask = iris.target == class_idx
    
    # Scatter plot
    ax.scatter(X_pca_2d[mask, 0],    # PC1 values
               X_pca_2d[mask, 1],    # PC2 values
               c=color, 
               marker=marker,
               label=name.capitalize(),
               s=80, 
               alpha=0.7,
               edgecolors='white',
               linewidth=0.5)

# Labels and title
ax.set_xlabel(f'Principal Component 1 ({explained_variance_ratio[0]*100:.1f}% variance)', fontsize=12)
ax.set_ylabel(f'Principal Component 2 ({explained_variance_ratio[1]*100:.1f}% variance)', fontsize=12)
ax.set_title('PCA Visualization of Iris Dataset\n(4D → 2D)', fontsize=14)

# Legend
ax.legend(title='Species', fontsize=10, title_fontsize=11)

# Grid
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.5)
ax.axvline(x=0, color='gray', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.show()

print("\n✓ With just 2 components, we capture {:.1f}% of variance!".format(
    cumulative_variance[1]*100))

---

## 8. IMPORTANT: PCA Limitation

### Theory

**THIS IS CRITICAL FOR STUDENTS TO UNDERSTAND!**

PCA has a fundamental limitation:

**PCA is UNSUPERVISED**

What this means:
- PCA does NOT use class labels
- It only finds directions of **maximum variance**
- It does NOT try to maximize **class separation**

### Why This Matters
The directions of maximum variance might NOT be the same as directions that separate classes!

### Example
Imagine a dataset where:
- Class A is spread widely in direction 1
- Class B is centered at a different point
- PCA will pick direction 1 (most variance)
- But direction 2 might separate A and B better!

### The Fix
For **supervised** dimensionality reduction, use **LDA** (Linear Discriminant Analysis):
- LDA uses class labels
- LDA maximizes class separation

---

In [ ]:
# =============================================================================
# SECTION 8: PCA LIMITATION DEMONSTRATION
# =============================================================================

print("=" * 60)
print("⚠️  IMPORTANT: PCA LIMITATION")
print("=" * 60)

print("""
PCA is UNSUPERVISED:
- It does NOT use class labels
- It only finds maximum VARIANCE directions
- It does NOT maximize CLASS SEPARATION

This is why PCA visualization might show class overlap!
""")

# Analyze class separation in PCA space
print("-" * 60)
print("CLASS CENTERS IN PCA SPACE")
print("-" * 60)

class_centers = {}
for i, name in enumerate(iris.target_names):
    mask = iris.target == i
    pc1_mean = np.mean(X_pca_2d[mask, 0])
    pc2_mean = np.mean(X_pca_2d[mask, 1])
    class_centers[name] = (pc1_mean, pc2_mean)
    print(f"  {name.capitalize():12}: PC1 = {pc1_mean:7.2f}, PC2 = {pc2_mean:7.2f}")

# Calculate distances between class centers
print("\n" + "-" * 60)
print("DISTANCE BETWEEN CLASS CENTERS")
print("-" * 60)

names = list(class_centers.keys())
for i in range(3):
    for j in range(i+1, 3):
        c1 = class_centers[names[i]]
        c2 = class_centers[names[j]]
        distance = np.sqrt((c1[0]-c2[0])**2 + (c1[1]-c2[1])**2)
        print(f"  {names[i]:10} ↔ {names[j]:10}: {distance:.2f}")

print("\n" + "=" * 60)
print("KEY TAKEAWAY")
print("=" * 60)
print("""
Setosa (blue) is well separated from others!
But Versicolor and Virginica overlap in PCA space.

This happens because PCA maximizes variance, NOT class separation.

→ For supervised separation, use LDA (Linear Discriminant Analysis)
""")

---

## Complete PCA Function

Here's all the code together in one function:

In [ ]:
# =============================================================================
# COMPLETE PCA IMPLEMENTATION
# =============================================================================

def pca_from_scratch(X, variance_threshold=0.95):
    """
    Implement PCA from scratch.
    
    Parameters:
    -----------
    X : np.array
        Data matrix of shape (n_samples, n_features)
    variance_threshold : float
        Keep components until this fraction of variance is explained
        Default: 0.95 (95%)
    
    Returns:
    --------
    X_transformed : np.array
        Transformed data in PCA space
    components : np.array
        Principal component vectors (eigenvectors)
    explained_variance : np.array
        Eigenvalues (variance for each component)
    explained_variance_ratio : np.array
        Fraction of total variance for each component
    """
    
    # Step 1: Mean centering
    # PCA requires centered data (mean = 0)
    means = np.mean(X, axis=0)
    X_centered = X - means
    
    # Step 2: Compute covariance matrix
    # C = (1/(n-1)) * X^T * X
    n = X_centered.shape[0]
    cov_matrix = (X_centered.T @ X_centered) / (n - 1)
    
    # Step 3: Eigenvalue decomposition
    # Find eigenvectors and eigenvalues
    eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
    eigenvalues = np.real(eigenvalues)
    eigenvectors = np.real(eigenvectors)
    
    # Step 4: Sort by eigenvalue (descending)
    # Largest eigenvalue = most variance = most important component
    sorted_idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[sorted_idx]
    eigenvectors = eigenvectors[:, sorted_idx]
    
    # Step 5: Calculate explained variance
    total_variance = np.sum(eigenvalues)
    explained_variance_ratio = eigenvalues / total_variance
    cumulative_variance = np.cumsum(explained_variance_ratio)
    
    # Determine number of components to keep
    n_components = np.argmax(cumulative_variance >= variance_threshold) + 1
    
    # Step 6: Select top n_components
    components = eigenvectors[:, :n_components]
    
    # Step 7: Transform data
    # Z = X_centered * W
    X_transformed = X_centered @ components
    
    return X_transformed, components, eigenvalues[:n_components], explained_variance_ratio[:n_components]


# ============================================================================
# TEST THE FUNCTION
# ============================================================================

print("=" * 60)
print("TESTING COMPLETE PCA FUNCTION")
print("=" * 60)

# Run PCA
X_pca_result, components, exp_var, exp_var_ratio = pca_from_scratch(X)

print(f"\nOriginal data shape: {X.shape}")
print(f"Transformed data shape: {X_pca_result.shape}")
print(f"Number of components used: {components.shape[1]}")
print(f"\nExplained variance ratios:")
for i, ratio in enumerate(exp_var_ratio):
    print(f"  PC{i+1}: {ratio*100:.1f}%")

# Compare with sklearn
from sklearn.decomposition import PCA
pca_sklearn = PCA(n_components=0.95)
X_sklearn = pca_sklearn.fit_transform(X)

print("\n" + "-" * 60)
COMPARISON WITH SKLEARN
print("-" * 60)
print(f"Our implementation: {X_pca_result.shape}")
print(f"sklearn PCA: {X_sklearn.shape}")
print(f"Results match: {np.allclose(np.abs(X_pca_result), np.abs(X_sklearn))}")

---

## Summary

### 8 Key Concepts Covered

1. ✅ **Data as Matrix** - $X \in \mathbb{R}^{n \times d}$
2. ✅ **Mean Centering** - $X_{centered} = X - \mu$
3. ✅ **Covariance Matrix** - $C = \frac{1}{n-1}X^TX$
4. ✅ **Eigenvalues/Eigenvectors** - $Cv = \lambda v$
5. ✅ **Explained Variance** - $EVR = \lambda / \sum\lambda$
6. ✅ **Projection** - $Z = X_{centered}W$
7. ✅ **Visualization** - 2D scatter plots
8. ✅ **Limitation** - Unsupervised (no class labels)

### Python Skills Used
- NumPy: `mean`, `var`, `@` (matrix multiply), `linalg.eig`
- Matplotlib: `scatter`, `plot`, `bar`, `axhline`
- Indexing and slicing
- Broadcasting

---

## Exercises for Students

1. Apply PCA to a different dataset (e.g., wine dataset)
2. Compare results with sklearn's PCA implementation
3. Visualize data in 3D using first 3 principal components
4. **Explain**: Why might PCA not separate classes well? What's the alternative?
5. **Research**: What is the relationship between PCA and SVD?

**End of Tutorial 4**